# Multi_FraudGT_HGB 결합기 학습·검증·Test

기본 모드는 완료된 전체 결과 조회입니다. 실제 재학습은 외부 점수 준비 후 `RUN_EXPERIMENT=True`로 켭니다. 기반 GNN/HGB 가중치는 고정합니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "gnn_hgb_fusion"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "gnn_hgb_fusion"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
MODEL_KEY = 'multi'
RESULT_KEY = 'multi_fraudgt_hgb'
RANDOM_STATE = 20261108
EXTERNAL = ROOT / 'external_data'
RUN_EXPERIMENT = False
from fusion_workflow import load_scores, check_disjoint, fit_and_freeze, evaluate_frozen_test, predict_logits
recorded_parameters = json.loads((ROOT / 'results' / RESULT_KEY / 'fusion_parameters.json').read_text())
display(pd.DataFrame([recorded_parameters]))

## 1. ID 정렬과 모집단 확인

Fit 7,315,599건, Validation 8,551,015건, Test 33,429,507건. HGB 점수와 GNN 점수는 ID로 정렬합니다. 각 구간 간 중복 ID가 없어야 합니다.

In [ ]:
if RUN_EXPERIMENT:
    fit = load_scores(EXTERNAL / MODEL_KEY / 'fusion_fit', 7315599)
    val = load_scores(EXTERNAL / MODEL_KEY / 'validation', 8551015)
    test = load_scores(EXTERNAL / MODEL_KEY / 'test', 33429507)
    check_disjoint(fit, val, test)
else:
    print('저장 결과 조회 모드: 외부 점수는 읽지 않습니다.')

## 2. 결합 학습과 Validation 임계값 고정

StandardScaler + LogisticRegression(C=0.1, lbfgs, max_iter=1000, tol=1e-7). class/sample weight와 추가 logit 보정은 없습니다. Validation F1 동률이면 더 높은 임계값을 고릅니다.

In [ ]:
from datetime import datetime, timezone
if RUN_EXPERIMENT:
    output = ROOT / 'outputs' / (MODEL_KEY + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    parameters, frozen = fit_and_freeze(fit, val, output, RANDOM_STATE)
    display(pd.DataFrame([frozen['primary']]))
else:
    frozen = json.loads((ROOT / 'results' / RESULT_KEY / 'frozen_validation.json').read_text())
    display(pd.DataFrame([frozen['thresholds']]))

## 3. Test 평가

위에서 고정한 결합기와 Validation 임계값으로만 평가합니다. 주 비교는 0.5입니다. 저장된 원본 결과와 새 실행 결과를 섞지 않습니다.

In [ ]:
if RUN_EXPERIMENT:
    result = evaluate_frozen_test(test, output)
    display(pd.DataFrame(result).T)
else:
    saved = pd.read_csv(ROOT / 'results' / RESULT_KEY / 'metrics.csv')
    display(saved)

전체 점수 생성·추론의 원본 코드는 `reference_source/`에 있습니다. 원본부터 재현하려면 데이터와 피처/그래프 캐시 및 기반 모델 가중치가 필요합니다.